##### For extracting images from zip file

In [1]:
import shutil
shutil.unpack_archive("Images.zip")

In [2]:
# %pip install tensorflow

In [3]:
import pandas as pd
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

2025-10-12 10:55:31.550545: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
labels_df = pd.read_csv("labels.csv", sep=",", header=None)

In [5]:
labels_df.head()

,0,1,2,3,4,5
0,0,pickup_truck,213,34,255,50
1,0,car,194,78,273,122
2,0,car,155,27,183,35
3,0,articulated_truck,43,25,109,55
4,0,car,106,32,124,45


In [6]:
labels_df.columns = ['image_id', 'class', 'xmin', 'ymin', 'xmax', 'ymax']

In [7]:
labels_df.head(20)

,image_id,class,xmin,ymin,xmax,ymax
0,0,pickup_truck,213,34,255,50
1,0,car,194,78,273,122
2,0,car,155,27,183,35
3,0,articulated_truck,43,25,109,55
4,0,car,106,32,124,45
5,1,bus,205,155,568,314
6,1,bus,285,123,477,168
7,1,car,544,162,617,193
8,1,car,329,152,371,163
9,1,car,447,161,497,183


In [8]:
labels_df["image_id"] = labels_df["image_id"].apply(lambda x:f"{x:08d}")

In [9]:
labels_df.head(7)

,image_id,class,xmin,ymin,xmax,ymax
0,00000000,pickup_truck,213,34,255,50
1,00000000,car,194,78,273,122
2,00000000,car,155,27,183,35
3,00000000,articulated_truck,43,25,109,55
4,00000000,car,106,32,124,45
5,00000001,bus,205,155,568,314
6,00000001,bus,285,123,477,168


##### Loading first 1000 images into array names images 

In [10]:
labels_df = labels_df.iloc[:1000]


images_dir = "Images/"

images228 = []
images480 = []

for index, row in labels_df.iterrows():
    img_path = os.path.join(images_dir, f"{row['image_id']}.jpg")
    img = cv2.imread(img_path)
    if img is not None:
        if len(img) == 228:
            images228.append(img)
        else:
            images480.append(img)
    else:
        print(f"Image {img_path} not found or could not be opened.")



In [11]:
len(images228) + len(images480)

1000

In [12]:
images228 = np.array(images228)

In [13]:
images480 = np.array(images480)

In [14]:
if len(images228) ==0:
    print("No images with height 228 found.")
else:
    print(f"Images with height 228 shape: {images228.shape} count: {len(images228)}")

Images with height 228 shape: (206, 228, 342, 3) count: 206


In [15]:
if len(images480) ==0:
    print("No images with height 228 found.")
else:
    print(f"Images with height 228 shape: {images480.shape} count: {len(images480)}")
    print(f"{len(images480)} images loaded successfully ")

Images with height 228 shape: (794, 480, 720, 3) count: 794
794 images loaded successfully 


In [16]:
## Analyze the distribution of vehicle types in the dataset
vehicle_types = labels_df['class'].value_counts()
print(vehicle_types)

class
car                      682
pickup_truck             111
motorized_vehicle         61
articulated_truck         30
work_van                  29
bus                       28
pedestrian                23
single_unit_truck         18
bicycle                   12
non-motorized_vehicle      5
motorcycle                 1
Name: count, dtype: int64


In [17]:
if len(images228)>0:
    processed_images = [cv2.resize(img, (224, 224)) for img in images228]

In [18]:
if len(images480)>0:
    processed_images += [cv2.resize(img, (224, 224)) for img in images480]

In [34]:
print(len(processed_images))
print(processed_images[0].shape)
type(processed_images)

1000
(224, 224, 3)


numpy.ndarray

In [ ]:
# processed_images = np.array(processed_images)
# type(processed_images)

numpy.ndarray

In [35]:
len(processed_images)

1000

In [21]:
labels_df.head()


,image_id,class,xmin,ymin,xmax,ymax
0,00000000,pickup_truck,213,34,255,50
1,00000000,car,194,78,273,122
2,00000000,car,155,27,183,35
3,00000000,articulated_truck,43,25,109,55
4,00000000,car,106,32,124,45


In [22]:
labels  = labels_df['class'].to_numpy()
bounding_boxes = labels_df[['xmin', 'ymin', 'xmax', 'ymax']].to_numpy()

In [23]:
print(labels[3])

articulated_truck


In [24]:
bounding_boxes[3]

array([ 43,  25, 109,  55])

In [25]:
np.unique(labels)


array(['articulated_truck', 'bicycle', 'bus', 'car', 'motorcycle',
       'motorized_vehicle', 'non-motorized_vehicle', 'pedestrian',
       'pickup_truck', 'single_unit_truck', 'work_van'], dtype=object)

In [26]:
# # Concert lables to numerical values : One Hot Encoding
# from sklearn.preprocessing import OneHotEncoder
# encoder = OneHotEncoder(sparse_output=False)
# labels_encoded = encoder.fit_transform(labels.reshape(-1, 1))
# labels_encoded[3]
# labels_encoded.shape

In [43]:
# Convert labels to label encoding -
labels = labels_df['class'].astype(str).to_numpy()  # Ensure all labels are strings
unique_labels = np.unique(labels)
print(f"Total Unique Labels are - {len(unique_labels)}")



Total Unique Labels are - 11


In [44]:
label_to_index = {label: index for index, label in enumerate(unique_labels)}
label_to_index

{'articulated_truck': 0,
 'bicycle': 1,
 'bus': 2,
 'car': 3,
 'motorcycle': 4,
 'motorized_vehicle': 5,
 'non-motorized_vehicle': 6,
 'pedestrian': 7,
 'pickup_truck': 8,
 'single_unit_truck': 9,
 'work_van': 10}

In [45]:
labels_encoded_int = np.array([label_to_index[label] for label in labels])
labels_encoded_int

array([ 8,  3,  3,  0,  3,  2,  2,  3,  3,  3,  3,  5,  3,  3,  2,  3,  0,
        3,  3,  8,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,
        8,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  2,  3,  3,  8,  0, 10,
        5,  3,  5,  3,  3,  3,  0,  3,  3,  3,  3,  3,  2,  2,  3,  5,  3,
        3,  3,  3,  5,  8,  3,  8,  3,  3,  8,  3,  3,  3,  3,  9,  9,  3,
        8,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  0,  3,  3,  3,  8,  3,
        3,  9,  3,  3,  8,  3,  3,  8,  8,  3,  3,  3,  3,  3,  3,  3,  3,
        3,  3,  3,  5,  3, 10,  3,  3,  2,  3,  3,  3, 10,  8,  3,  3,  3,
        3,  3,  3,  3,  3,  3,  3,  3,  5,  7,  3,  3,  5,  3,  8,  3,  3,
        8,  3,  3,  3,  3,  3,  5,  3,  8,  8, 10,  8,  8,  3,  8,  3,  3,
        3,  5,  3,  5,  8,  3,  5,  8,  8,  3,  3, 10,  0,  5,  3,  8,  3,
        3,  8,  3,  8,  3,  3,  5,  3,  3,  3,  8,  8,  3,  8, 10,  3,  3,
        8,  3,  8,  3,  3,  3,  3,  3,  3,  3,  3,  3,  5,  0,  3,  3,  3,
        3,  3,  3,  3,  3

In [65]:
labels = np.array([label_to_index[label] for label in labels])

In [66]:
len(labels)

1000

In [49]:
len(bounding_boxes)

1000

In [48]:
labels_df.head()

,image_id,class,xmin,ymin,xmax,ymax
0,00000000,pickup_truck,213,34,255,50
1,00000000,car,194,78,273,122
2,00000000,car,155,27,183,35
3,00000000,articulated_truck,43,25,109,55
4,00000000,car,106,32,124,45


In [71]:
X_train, X_test, y_train, y_test, bbox_train, bbox_test = train_test_split(processed_images, labels, bounding_boxes, test_size=0.2, random_state=42)

# Creation of CNN Architecture 

In [72]:
# Model Creation
def create_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.Flatten()(x)
    
    x = layers.Dense(64, activation='relu')(x)

    vehicle_class = layers.Dense(num_classes, activation='softmax', name='vehicle_class')(x)

    bounding_box = layers.Dense(4, name='bounding_box')(x)
    model = keras.Model(inputs=inputs, outputs=[vehicle_class, bounding_box])
    return model

In [73]:
input_shape = processed_images[0].shape
num_classes = len(unique_labels)

In [74]:
model = create_model(input_shape, num_classes)

In [75]:
model.compile(optimizer='adam', loss={'vehicle_class': 'sparse_categorical_crossentropy', 'bounding_box': 'mse'}, 
                metrics={'vehicle_class': 'accuracy', 'bounding_box': 'mse'})

In [81]:
model.fit(X_train, {"vehicle_class":y_train, "bounding_box": bbox_train}, epochs=100, validation_data=(X_test, {"vehicle_class":y_test, "bounding_box": bbox_test}))

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 31s 1s/step - bounding_box_loss: 32487.8105 - bounding_box_mse: 32487.8105 - loss: 32765.7168 - vehicle_class_accuracy: 0.0125 - vehicle_class_loss: 277.9120 - val_bounding_box_loss: 35440.6133 - val_bounding_box_mse: 33963.3086 - val_loss: 34109.8555 - val_vehicle_class_accuracy: 0.0100 - val_vehicle_class_loss: 137.0612
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 33s 1s/step - bounding_box_loss: 29500.8730 - bounding_box_mse: 29500.8730 - loss: 29636.1719 - vehicle_class_accuracy: 0.3063 - vehicle_class_loss: 135.3043 - val_bounding_box_loss: 30575.9551 - val_bounding_box_mse: 29789.4727 - val_loss: 29906.8926 - val_vehicle_class_accuracy: 0.6850 - val_vehicle_class_loss: 104.8376
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 33s 1s/step - bounding_box_loss: 27137.7598 - bounding_box_mse: 27137.7598 - loss: 27248.9473 - vehicle_class_accuracy: 0.5612 - vehicle_class_loss: 111.1857 - val_bounding_box_loss: 30252.7383 - val_bounding_box_mse: 30020.7246 - val_

## Module Evaluation

In [82]:
model.evaluate(X_test, {"vehicle_class": y_test, "bounding_box": bbox_test}, verbose=2)

7/7 - 2s - 284ms/step - bounding_box_loss: 30907.3535 - bounding_box_mse: 30019.3379 - loss: 30021.4922 - vehicle_class_accuracy: 0.5500 - vehicle_class_loss: 1.9751


[30021.4921875,
 1.9750902652740479,
 30907.353515625,
 30019.337890625,
 0.550000011920929]

In [83]:
sample_image = X_test[:5]
predictions = model.predict(sample_image)
predicted_bounding_boxes = predictions[1]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
